# ToolGuard: quick start
Support-ticket agents with flaky tools and noisy input: does a guard layer keep them safe?

Dataset: *ToolGuard: Tool-Using Agent Reliability Benchmark*. Code: [github.com/MHKhan50/agentic-ai-research-log](https://github.com/MHKhan50/agentic-ai-research-log) · Software DOI: [10.5281/zenodo.24230498](https://doi.org/10.5281/zenodo.24230498)

Data licence: CC BY 4.0 (credit Muzammal Hussain).

## 1. Load every file
Attach the dataset first (right panel → **Add Input** → **Datasets**). The cell finds the files wherever Kaggle mounts them.

In [ ]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt

BLUE = "#1F4E9C"
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

def data_files():
    """Every data file in the attached dataset (works for any folder layout)."""
    out = []
    for root in ("/kaggle/input", "data"):
        for d, _, files in os.walk(root, followlinks=True):
            out += [os.path.join(d, f) for f in sorted(files)
                    if f.lower().endswith((".csv", ".jsonl", ".json", ".parquet"))]
        if out:
            return out
    raise FileNotFoundError("No data files found. Attach the dataset: right panel -> Add Input -> Datasets.")

def load(path):
    if path.endswith(".csv"):
        return pd.read_csv(path, low_memory=False)
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    if path.endswith(".jsonl"):
        return pd.read_json(path, lines=True)
    obj = json.load(open(path, encoding="utf-8"))
    if isinstance(obj, list) and obj and isinstance(obj[0], dict):
        return pd.json_normalize(obj)
    return None  # metadata or summary file, not a table

tables = {}
for p in data_files():
    t = load(p)
    if t is not None:
        tables[os.path.basename(p)] = t
print(len(tables), "tables loaded")

## 2. What is in the dataset

In [ ]:
overview = pd.DataFrame(
    [(name, len(t), t.shape[1], ", ".join(map(str, t.columns[:8])) + (" ..." if t.shape[1] > 8 else ""))
     for name, t in tables.items()],
    columns=["file", "rows", "columns", "first columns"])
overview

## 3. First rows of the main table

In [ ]:
main = max(tables, key=lambda k: len(tables[k]))
print("Largest table:", main)
tables[main].head()

## 4. One key figure

In [ ]:
OPTIONS = [('policy', 'success_rate', 'mean', 'Task success rate, by policy', 'Success rate', True), ('policy', 'success', 'mean', 'Task success rate, by policy', 'Success rate', True), ('category', None, 'count', 'Tickets by category', 'Tickets', False)]

def pick():
    for x, y, agg, title, ylabel, pct in OPTIONS:
        need = {x} | ({y} if y else set())
        hits = [(n, t) for n, t in tables.items() if need <= set(t.columns)]
        if hits:
            same = [t for n, t in hits if list(t.columns) == list(hits[0][1].columns)]
            name = hits[0][0] if len(same) == 1 else f"{len(same)} files with the same columns"
            return name, pd.concat(same, ignore_index=True), x, y, agg, title, ylabel, pct
    return None

found = pick()
if found is None:
    print("The columns used for this chart are not in the attached files.")
else:
    name, t, x, y, agg, title, ylabel, pct = found
    if agg == "year_count":
        s = pd.to_datetime(t[x], errors="coerce").dt.year.value_counts().sort_index()
        fig, ax = plt.subplots(figsize=(9, 3.4))
        ax.bar(s.index, s.values, color=BLUE, width=0.8)
        ax.set_xlabel("Year")
    else:
        if agg == "count":
            s = t[x].value_counts().sort_values()
        else:
            v = t[y]
            if v.dtype == object:
                v = v.map({"True": 1, "False": 0, "true": 1, "false": 0}).fillna(pd.to_numeric(v, errors="coerce"))
            s = pd.to_numeric(v, errors="coerce").groupby(t[x].astype(str)).mean().dropna().sort_values()
        fig, ax = plt.subplots(figsize=(8, 0.5 * len(s) + 1.2))
        ax.barh(s.index.astype(str), s.values, color=BLUE)
        for i, val in enumerate(s.values):
            ax.text(val, i, f"  {val:.1%}" if pct else f"  {val:,.2f}".rstrip("0").rstrip("."), va="center")
        ax.margins(x=0.15)
    ax.set_title(title, loc="left")
    ax.set_ylabel(ylabel if agg == "year_count" else "")
    ax.spines[["top", "right"]].set_visible(False)
    plt.tight_layout(); plt.show()
    print("Source table:", name)

## 5. Next steps
The full experiment, the column definitions and the limitations are in the GitHub repository above. If you use the data, please cite it with the DOI.